# Days-of-Week Cyclic Geometry
Replicating the 3D ring structure from *Manifold Steering Reveals the Shared Geometry of Neural Network Representation and Behavior*.

**Pipeline:** Extract Llama 3.1 8B layer-28 activations → 64D PCA → periodic spline → 3D PCA → interactive plot.

In [ ]:
%pip install nnsight plotly tqdm -q

In [ ]:
import numpy as np
import time
import colorsys
import os
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
from scipy.interpolate import CubicSpline
import plotly.graph_objects as go
from huggingface_hub import login as hf_login
import nnsight
from nnsight import LanguageModel

## Setup

In [ ]:
import os

HF_TOKEN  = os.environ.get("HF_TOKEN", "")   # set in your shell: export HF_TOKEN=hf_...
API_KEY   = os.environ.get("NDIF_API_KEY", "")  # set in your shell: export NDIF_API_KEY=...
MODEL_ID  = "meta-llama/Meta-Llama-3.1-8B"
LAYER_IDX = 28

hf_login(token=HF_TOKEN, add_to_git_credential=False)
nnsight.CONFIG.set_default_api_key(API_KEY)
model = LanguageModel(MODEL_ID)
print(f"Model ready: {MODEL_ID}")

## Prompt Generation

Two templates × 7 starting days × 6 k-values = **84 prompts**.  
Each prompt is labeled by its *correct answer day* (not the starting day).  
This gives 12 prompts per answer-day — enough for stable centroids and full 64D PCA.

In [ ]:
DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

templates = [
    lambda day, k: f"What day is {k} days after {day}?",
    lambda day, k: f"If today is {day}, what day will it be in {k} days?",
]

prompts, labels = [], []
for day_idx, day in enumerate(DAYS):
    for k in range(1, 7):
        for tmpl in templates:
            prompts.append(tmpl(day, k))
            labels.append(DAYS[(day_idx + k) % 7])
labels = np.array(labels)

print(f"Total prompts: {len(prompts)}")
print(f"Prompts per answer-day: {[(d, (labels == d).sum()) for d in DAYS]}")
print("\nFirst 4 examples:")
for i in range(4):
    print(f"  '{prompts[i]}' → {labels[i]}")

## Activation Extraction

Extract layer-28 hidden states at the **last token position** for each prompt.  
Results are cached to `all_acts.npy` — skip this cell on re-runs and load from cache instead.

In [ ]:
CACHE_ACTS = Path("all_acts.npy")
CACHE_LABELS = Path("labels.npy")

if CACHE_ACTS.exists():
    print("Cache found — loading from disk (skip API calls).")
    all_acts = np.load(CACHE_ACTS)
    labels = np.load(CACHE_LABELS, allow_pickle=True)
    print(f"Loaded: all_acts={all_acts.shape}, labels={labels.shape}")
else:
    print(f"No cache found. Extracting activations for {len(prompts)} prompts...")
    all_acts = []

    for i, prompt in enumerate(tqdm(prompts, desc="Extracting layer-28 hidden states")):
        for attempt in range(3):
            try:
                # nnsight 0.7: output is already the hidden state tensor [batch, seq, 4096]
                with model.trace(prompt, remote=True):
                    hidden = model.model.layers[LAYER_IDX].output[:, -1, :].save()
                act = hidden.squeeze(0).cpu().float().numpy()
                all_acts.append(act)
                break
            except Exception as e:
                print(f"  Prompt {i} attempt {attempt+1} failed: {e}")
                if attempt < 2:
                    time.sleep(5)
                else:
                    raise
        time.sleep(0.25)

    all_acts = np.stack(all_acts)
    np.save(CACHE_ACTS, all_acts)
    np.save(CACHE_LABELS, labels)
    print(f"Saved to disk. Shape: {all_acts.shape}")

## PCA Step 1 — Intermediate 64D Space

Fit PCA on all 84 individual activations (not just the 7 centroids).  
This finds a richer subspace that captures both between-day and within-day variation.

In [ ]:
N_INTER = min(64, len(prompts))  # 64 with 84 prompts; guards against edge cases

pca1 = PCA(n_components=N_INTER)
acts_inter = pca1.fit_transform(all_acts)  # (84, 64)

cumvar = np.cumsum(pca1.explained_variance_ratio_)
print(f"Intermediate space: {acts_inter.shape}")
print(f"Variance explained by top 64 PCs: {cumvar[-1]:.3f}")
print(f"Variance explained by top  7 PCs: {cumvar[6]:.3f}")
print(f"Variance explained by top  3 PCs: {cumvar[2]:.3f}")

## Compute Per-Day Centroids

Average activations grouped by *answer day* — all prompts whose correct answer is "Wednesday" average together to form the Wednesday centroid, etc.

In [ ]:
centroids = np.array([
    acts_inter[labels == day].mean(axis=0)
    for day in DAYS
])  # (7, 64)

print(f"Centroids shape: {centroids.shape}")
for day in DAYS:
    n = (labels == day).sum()
    print(f"  {day}: averaged from {n} prompts")

## Periodic Cubic Spline in 64D Space

Fit a smooth closed curve through the 7 centroids in their natural cyclic order.  
`bc_type='periodic'` enforces that derivatives match at the Sunday→Monday wraparound.

In [ ]:
# Append Monday at the end to satisfy scipy's periodic boundary requirement (y[0] == y[-1])
centroids_periodic = np.vstack([centroids, centroids[0:1]])  # (8, 64)
t_knots = np.arange(8, dtype=float)  # [0, 1, 2, 3, 4, 5, 6, 7]
                                      # Mon=0, Tue=1, ..., Sun=6, Mon(wrap)=7

spline = CubicSpline(t_knots, centroids_periodic, bc_type='periodic')

# Sample 300 points along the full cycle
t_dense = np.linspace(0, 7, 300)
curve_inter = spline(t_dense)  # (300, 64)

print(f"Spline curve sampled: {curve_inter.shape}")
print(f"Curve closes back: start≈end distance = {np.linalg.norm(curve_inter[0] - curve_inter[-1]):.4f}")

## PCA Step 2 — Project Manifold to 3D

**Critical:** `pca2` is fit on the *spline curve samples*, not on raw activations.  
This ensures the 3 principal components capture the manifold's own geometric directions, so the 3D plot directly reveals the ring shape.

In [ ]:
pca2 = PCA(n_components=3)
curve_3d = pca2.fit_transform(curve_inter)   # (300, 3)
centroids_3d = pca2.transform(centroids)      # (7, 3)

var = pca2.explained_variance_ratio_
print(f"Curve 3D shape:     {curve_3d.shape}")
print(f"Centroids 3D shape: {centroids_3d.shape}")
print(f"\nVariance per component: PC1={var[0]:.3f}  PC2={var[1]:.3f}  PC3={var[2]:.3f}")
print(f"Total manifold variance in 3D: {var.sum():.3f}")
print()
if var[:2].sum() > 0.8:
    print("PC1+PC2 > 80% → strong planar ring structure")
elif var[:2].sum() > 0.6:
    print("PC1+PC2 > 60% → moderate ring structure")
else:
    print("Low variance in top 2 PCs → more complex geometry, check the 3D plot")

## Interactive 3D Visualization

In [ ]:
# Cyclic HSV palette — Mon and Sun get adjacent hues to reflect their adjacency on the ring
day_colors = [
    '#{:02x}{:02x}{:02x}'.format(*[int(c * 255) for c in colorsys.hsv_to_rgb(i / 7, 0.85, 0.88)])
    for i in range(7)
]

# Map t_dense (0..7) to a 0..1 color scale for the continuous curve
colorscale = [[i / 6, day_colors[i]] for i in range(7)]

fig = go.Figure()

# --- Spline manifold curve ---
fig.add_trace(go.Scatter3d(
    x=curve_3d[:, 0],
    y=curve_3d[:, 1],
    z=curve_3d[:, 2],
    mode='lines',
    name='Manifold Curve',
    line=dict(
        color=t_dense,
        colorscale=colorscale,
        width=6,
        showscale=False,
    ),
    hovertemplate='t=%.2f<extra>Spline</extra>',
))

# --- Day centroids ---
fig.add_trace(go.Scatter3d(
    x=centroids_3d[:, 0],
    y=centroids_3d[:, 1],
    z=centroids_3d[:, 2],
    mode='markers+text',
    name='Day Centroids',
    text=DAYS,
    textposition='top center',
    textfont=dict(size=13, color='black', family='Arial Black'),
    marker=dict(
        size=10,
        color=day_colors,
        symbol='circle',
        line=dict(color='black', width=1.5),
    ),
    hovertemplate='%{text}<br>PC1=%{x:.3f} PC2=%{y:.3f} PC3=%{z:.3f}<extra></extra>',
))

fig.update_layout(
    title=dict(
        text=(
            f'Days-of-Week Cyclic Ring — Llama 3.1 8B, Layer {LAYER_IDX}<br>'
            f'<sup>Multi-step PCA: 4096D → 64D → Periodic Spline → 3D | '
            f'{len(prompts)} prompts, {len(DAYS)} centroids</sup>'
        ),
        x=0.5,
        font=dict(size=14),
    ),
    scene=dict(
        xaxis_title=f'PC1 ({var[0]*100:.1f}%)',
        yaxis_title=f'PC2 ({var[1]*100:.1f}%)',
        zaxis_title=f'PC3 ({var[2]*100:.1f}%)',
        aspectmode='cube',
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)),
    ),
    legend=dict(x=0.02, y=0.98),
    width=900,
    height=720,
    margin=dict(t=100, b=20, l=20, r=20),
)

fig.show()
fig.write_html("days_geometry_3d.html")
print("Saved interactive plot to days_geometry_3d.html")

## Sanity Checks

Verify the ring structure is genuine:

In [ ]:
# Pairwise distances between day centroids in 64D space
from scipy.spatial.distance import cdist

dist_matrix = cdist(centroids, centroids)
print("Pairwise distances between day centroids (64D):")
print("       " + "  ".join(f"{d[:3]:>4}" for d in DAYS))
for i, day in enumerate(DAYS):
    row = "  ".join(f"{dist_matrix[i, j]:6.3f}" for j in range(7))
    print(f"{day[:3]}: {row}")

print("\nAdjacent-day distances (should be smaller than opposite-day distances):")
for i in range(7):
    adj = dist_matrix[i, (i+1) % 7]
    opp = dist_matrix[i, (i+3) % 7]
    print(f"  {DAYS[i]}→{DAYS[(i+1)%7]}: {adj:.3f}  |  {DAYS[i]}→{DAYS[(i+3)%7]}: {opp:.3f}")